In [ ]:
from frap_utils import *

## Generate list of paths

In [ ]:
# Create list of paths to the final files
sample_names = ["WT", "MGS1", "MGS2", "MGS3", "MGS4", "MGS5"]

# Path to main folder
root_folder = Path("/mnt/c/users/elopatukhin/Desktop/Data_processing/050326_U2OS_live_FRAP")

# Path to the folder to save final graphs
graphs_folder = check_dir_exists(root_folder / "final_graphs")

list_of_paths = []
for sample in sample_names:
    filepath = check_file_exists(root_folder / sample / "EasyFRAP" / f"easyFRAP_Final_Data_{sample}_MFI_filtrated.csv")
    list_of_paths.append(filepath)
    print(f"File: {filepath}")

print(f"There are {len(list_of_paths)} files founded.")

## Import data

In [ ]:
# Import data
dfs = []  # list of DataFrames

for p in list_of_paths:
    df = pd.read_csv(p, header=0)
    dfs.append(df)

print(f"Loaded {len(dfs)} files.")

## Histograms of nuclei MFI

In [ ]:
# Histograms of Nuclear MFI
for i, sample_name in zip(range(len(dfs)), sample_names):

    var = dfs[i]["mean_intensity"]
    plt.hist(var, bins = 15, edgecolor='black')
    plt.xlabel("Nucleur mean fluorescent intensity")
    plt.ylabel("Frequency")
    plt.title(sample_names[i])

    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()

    # Save image
    plt.savefig(graphs_folder/f"Nuclei_MFI_hist_{sample_name}.png", dpi=300, bbox_inches="tight")
    print(f"Plot for {sample_name} is saved in the directory: {graphs_folder}.")

    plt.show()

## *Optional: delete outliers*

In [ ]:
dfs[2] = dfs[2].drop(index=1)

## *Optional: extra filtration based on the nuclei MFI*

In [ ]:
threshold_MFI = 16
dfs_filtered = []
for i in range(len(dfs)):
    df_filtered = dfs[i][dfs[i]['mean_intensity'] < threshold_MFI]
    dfs_filtered.append(df_filtered)
    print(f"{i}: keep {len(df_filtered)} out of {len(dfs[i])} rows.")

## Boxplots

In [ ]:
args = [{'var': "T-half", 'ylabel':'Recovery time, sec', 'title':''},
        {'var': "Mobile Fraction", 'ylabel':'Mobile fraction, %', 'title': ''},
]

for arg in args:
    beautiful_boxplot(
        df_list=[
                dfs[0][arg['var']],
                dfs[1][arg['var']],
                dfs[2][arg['var']],
                dfs[3][arg['var']],
                dfs[4][arg['var']],
                dfs[5][arg['var']]
                ],
        labels=sample_names,
        ylabel=arg['ylabel'],
        title=arg['title'],
        show = True,
        save = True,
        name = arg['var'],
        output_dir = graphs_folder
    )

## Mann-Whitney statistical analysis

### Mobile fraction, %

In [ ]:
mannwhitneyu_stats(data = dfs,
                  column_name = 'Mobile Fraction',
                  reference_index = 0,
                  save = True,
                  output_folder = graphs_folder)

### Half-time of recovery, sec

In [ ]:
mannwhitneyu_stats(data = dfs,
                  column_name = 'T-half',
                  reference_index = 0,
                  save = True,
                  output_folder = graphs_folder)